# Assignment 1 — Dimensionality Reduction with Animal Faces
**MCS 3950 Computer Vision · UPEI · Fall 2026**

| | |
|---|---|
| **Released** | Wednesday, October 21, 2026 |
| **Due** | Monday, November 9, 2026 at 11:59 pm (Atlantic) |
| **Weight** | 10% of the course grade · **60 points**, all marked automatically |
| **Work** | Individual (see *Academic integrity* below) |
| **Submit** | Commit this notebook to your assignment repository |

In Lecture 9 we built PCA one step at a time on aerial-photo patches and on human faces. In this
assignment you package those steps into your own PCA class, check it against scikit-learn, and
then use it on **1,500 grayscale animal faces** (cats, dogs and wild animals) to see what a
linear "face space" captures, what it throws away, and where it fails.

| Exercise | Topic | Points |
|---|---|---|
| 0 | Mechanics: commits, clean run, no data committed, AI disclosure | 4 |
| 1 | Warm-up: notation, PCA by hand, reconstruction error | 12 |
| 2 | Implementing PCA with the SVD | 8 |
| 3 | Eigenfaces: choosing $k$, and inputs that do not belong | 16 |
| 4 | Interpreting the components | 16 |
| 5 | Challenge: searching in face space | 4 |

**Notation** (same as L9). $X$ is $n \times d$ (one flattened image per row), $\mu$ is the
mean row, $W$ is $k \times d$ (one principal component per row),
$Z = (X - \mu)W^\top$ and $\hat X = ZW + \mu$.

## How this assignment is marked

**All 60 points are marked by a program** that re-runs your notebook from top to bottom after
the deadline and checks the variables and functions named in each question. So:

* **Keep the names.** Each question lists its *graded variables*. Store your answer in exactly
  that name, with the type the question asks for. A correct answer under a different name scores 0.
* **Do not edit or delete the provided cells**, and do not change the random seeds.
* **The notebook must run from top to bottom** on a fresh kernel. A cell that crashes costs the
  points that depend on it, plus the clean-run point.
* Some questions have **self-check** cells that tell you whether you are right. Most do not:
  the marking program uses its own tests (for example, it tests your functions on data you have
  not seen), so test your work yourself.

**The 🧠 *Think about it* boxes are not marked**, but questions like them may appear on
Tests 2 and 3, so work through them: the marks for *understanding* this assignment come from
the tests, not from this notebook.

## Instructions

**Getting the notebook.** Your assignment repository was created when you accepted the
assignment. Work in **either** of these ways:

* **Google Colab.** In Colab choose *File → Open notebook → GitHub*, tick *Include private
  repos*, pick your Assignment 1 repository and open `A1_PCA_Animal_Faces.ipynb`. Save your work
  back with *File → Save a copy in GitHub*: same repository, branch `main`, **same file
  name**, and a short commit message saying what you did. Each save is one commit.
* **Locally.** `git clone` your repository, `pip install -r requirements.txt`, open the
  notebook in Jupyter or VS Code, and `git commit` / `git push` as you go.

**Data.** The first code cell downloads the images (14 MB) into a `data/` folder. Do not
commit that folder; the `.gitignore` already keeps it out.

**Before you submit:** *Runtime → Restart session and run all* (Colab) or *Restart & Run All*
(Jupyter), fix anything that crashes, fill in the AI disclosure at the end, and commit.

**Late work.** Up to 48 hours late: −35%. After 48 hours: not accepted without prior
arrangement. The marking program grades the last commit before the deadline; if you commit
again within 48 hours it also grades that version with the penalty and keeps the better mark.

**Academic integrity.** Discuss ideas with classmates, but the work you submit must be your own.
Generative AI tools are allowed **only if you disclose them** in the section at the end.

<div class="alert alert-info">

### Exercise 0 — Mechanics &nbsp;·&nbsp; *4 points*

Checked from your repository: at least three commits of your own (1); the notebook runs top to
bottom without errors (1); no data files committed (1); `ai_disclosure` filled in at the end (1).

</div>

## Setup

In [ ]:
import hashlib, os, urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11,
                     "image.cmap": "gray"})
GOLD, RED, TEAL, NAVY, GREY = "#C8962C", "#B82828", "#1A7A8A", "#1A3A5C", "#7A8FA6"

Helper functions. **Run this cell; do not change it.** It holds the self-check, the data
download, and plotting helpers used later.

In [ ]:
# ---------------------------------------------------------------- self-check
_KEYS = {
    "1.1": "832802ea4dd073d161333bf4747049cc02a76ddb",
    "1.2.1": "6d1982c55d749da008b690428e765924f5c56545",
    "1.2.2": "b3b1079bd1c90170640763eed13fe2c060d5607b",
    "1.2.3": "43393e1c6b812cbad50c782f2b75bf9e3d2b7cd5",
    "1.3.1": "85326a1f139961f25a41b82a77ed6a12f2043d66",
    "1.3.2": "afd54401d0bd08b469e87fdbbf3d86fa4d3ef8a5"
}

def _norm(v):
    if isinstance(v, str):
        return v.strip().upper()
    a = np.asarray(v, dtype=float)
    return ",".join(f"{x:.2f}" for x in np.round(a.ravel(), 2))

def check(qid, *values):
    """Self-check for question `qid`. Prints a verdict; awards no marks by itself."""
    try:
        h = hashlib.sha1("|".join(_norm(v) for v in values).encode()).hexdigest()
    except Exception as e:  # e.g. a value is still `...`
        print(f"{qid}: cannot check yet ({type(e).__name__})"); return
    print(f"{qid}: " + ("correct ✓" if h == _KEYS[qid] else "not yet ✗ — keep going"))

# ---------------------------------------------------------------- data
DATA_URL = "https://raw.githubusercontent.com/andrewgodbout/MCS-3950-F26/main/Notebooks/data/A1_animal_faces.npz"
DATA_SHA256 = "4fef07dd4e3ff302585b07cf98f59cb8d2300ed9b87f49cb1944a0f532a44d1d"

def load_animal_faces(path="data/A1_animal_faces.npz"):
    """Download (once) and return the faces as float32 in [0, 1], shape (1500, 100, 100)."""
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        print("downloading animal faces (14 MB) ...")
        urllib.request.urlretrieve(DATA_URL, path)
    digest = hashlib.sha256(open(path, "rb").read()).hexdigest()
    if digest != DATA_SHA256:
        os.remove(path)
        raise IOError("Download was corrupted or incomplete. Re-run this cell.")
    with np.load(path, allow_pickle=False) as f:
        return f["faces"].astype(np.float32) / 255.0

# ---------------------------------------------------------------- plotting
def show_images(images, titles=None, ncol=5, size=2.0, vmin=None, vmax=None, suptitle=None):
    """Show a list of 2-D arrays in a grid. Flattened 10,000-vectors are reshaped to 100x100."""
    images = [np.asarray(im).reshape(100, 100) if np.asarray(im).ndim == 1 else im
              for im in images]
    nrow = int(np.ceil(len(images) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(size * ncol, size * nrow + 0.3),
                             squeeze=False)
    for i, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if i < len(images):
            ax.imshow(images[i], vmin=vmin, vmax=vmax)
            if titles is not None:
                ax.set_title(titles[i], fontsize=9)
    if suptitle:
        fig.suptitle(suptitle)
    fig.tight_layout()
    plt.show()

def get_recon_error_df(orig, recon):
    """Per-row Euclidean reconstruction error, min-max normalised to [0, 1]."""
    loss = np.sqrt(np.sum((orig - recon) ** 2, axis=1))
    loss = (loss - np.min(loss)) / (np.max(loss) - np.min(loss))
    return pd.DataFrame(data=loss, columns=["recon_error"])

def plot_strong_comp_images(X_anims, Z, W, compn=1, image_shape=(100, 100),
                            positive_direction=True):
    """Component `compn` as an image, then the six faces chosen by YOUR `strongest`
    function (Exercise 4.1)."""
    inds = strongest(Z, compn, n=6, positive=positive_direction)
    fig, ax = plt.subplots(1, 7, figsize=(13, 2.4), subplot_kw={"xticks": (), "yticks": ()})
    lim = np.abs(W[compn]).max()
    ax[0].imshow(W[compn].reshape(image_shape), cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax[0].set_title(f"component {compn}", fontsize=9)
    for a, i in zip(ax[1:], inds[:6]):
        a.imshow(X_anims[i].reshape(image_shape))
        a.set_title(f"#{i}  z={Z[i, compn]:+.1f}", fontsize=8)
    fig.suptitle(f"component {compn}: most {'positive' if positive_direction else 'negative'} scores",
                 fontsize=10)
    fig.tight_layout()
    plt.show()

def img_tiling(images, tile_idx, size=8, image_shape=(100, 100)):
    """Plot a 5x5 tiling. tile_idx[i, j] is the image index for column i (left->right, first
    dimension increasing) and row j (bottom->top, second dimension increasing)."""
    tile_idx = np.asarray(tile_idx, dtype=int)
    plt.figure(figsize=(size, size))
    for i in range(5):
        for j in range(5):
            plt.imshow(images[tile_idx[i, j]].reshape(image_shape),
                       extent=(i * 32, (i + 1) * 32, j * 32, (j + 1) * 32), vmin=0, vmax=1)
    plt.xlim((0, 160)); plt.ylim((0, 160))
    plt.xticks([]); plt.yticks([])

print("helpers loaded")

---
## Exercise 1 — Warm-up

PCA is one of the oldest tools in data analysis — Pearson described it in 1901 — and one of
the first that computer vision borrowed: the *eigenfaces* of Turk and Pentland (1991) are PCA
on aligned face photographs. Before the faces, a few small exercises on the notation and on
what reconstruction error measures.

<div class="alert alert-info">

### 1.1 Dimensions &nbsp;·&nbsp; *3 points*

You apply PCA to a data matrix $X_{n\times d}$ ($n$ examples, $d$ features) and keep $k$
components. What are the dimensions of $W$ (the components), $Z$ (the low-dimensional
representation) and $\hat X$ (the reconstruction)? Choose from
(A) $n\times k$, (B) $n\times d$, (C) $k\times d$.

**Graded variables** (keep these names): `W_dim`, `Z_dim`, `X_hat_dim`

</div>

In [ ]:
# Assign each variable "A", "B" or "C"
W_dim = ...
Z_dim = ...
X_hat_dim = ...

In [ ]:
check("1.1", W_dim, Z_dim, X_hat_dim)  # self-check

### 1.2 PCA by hand

PCA was trained on an already-centred data matrix `X` (not shown), giving the `Z` and `W` below.

In [ ]:
Z = np.array([[10, 10], [5, 2], [4, 3], [4, 3]])
W = np.array([[0.5, 0.5, -0.5, -0.5], [0.7, 0.1, 0.7, 0.1]])
print("Z =\n", Z, "\nW =\n", W)

<div class="alert alert-info">

### 1.2.1 &nbsp;·&nbsp; *1 point*

How many rows and columns does the data matrix `X` have? (integers)

**Graded variables** (keep these names): `X_rows`, `X_cols`

</div>

In [ ]:
X_rows = ...
X_cols = ...

In [ ]:
check("1.2.1", X_rows, X_cols)  # self-check

<div class="alert alert-info">

### 1.2.2 &nbsp;·&nbsp; *1 point*

Here we reduce the dimensionality from $d$ = ___ to $k$ = ___. (integers)

**Graded variables** (keep these names): `d_orig`, `k_reduced`

</div>

In [ ]:
d_orig = ...      # original dimension
k_reduced = ...   # reduced dimension

In [ ]:
check("1.2.2", d_orig, k_reduced)  # self-check

<div class="alert alert-info">

### 1.2.3 &nbsp;·&nbsp; *2 points*

Find the low-dimensional representation `Z_new` of the (already centred) `X_new` below, as a
NumPy array. Use matrix operations, not hand arithmetic.

**Graded variables** (keep these names): `Z_new`

</div>

In [ ]:
X_new = np.array([[1, 1, 1, 1], [1, 0, 1, 1]])

In [ ]:
Z_new = ...
Z_new

In [ ]:
check("1.2.3", Z_new)  # self-check

<div class="alert alert-info">

### 1.2.4 Same code, same image? &nbsp;·&nbsp; *2 points*

Rows 3 and 4 of `Z` are identical, and so are rows 3 and 4 of `X`.

1. If two images have **identical** codes $Z$ after PCA (with $k < d$), must the images be
   identical? Set `same_code_answer` to `"A"`, `"B"` or `"C"`: *(1)*
   (A) Yes, always. (B) No: any difference orthogonal to every component disappears.
   (C) Only if the data were standardised first.
2. Prove it with an example: find a **non-zero** vector `v_same_code` (length 4, NumPy array)
   such that adding it to any row of `X` leaves that row's code unchanged. *(1)*
   *Hint: what must $Wv$ equal?*

**Graded variables** (keep these names): `same_code_answer`, `v_same_code`

</div>

In [ ]:
same_code_answer = ...
v_same_code = ...
print("W @ v =", W @ v_same_code)

<div class="alert alert-success">

🧠 **Think about it** — *not graded; questions like this may appear on Tests 2 and 3.*

PCA with $k < d$ is many-to-one. What does that mean for using $Z$ to decide whether two
images are "the same"? What kind of difference between two faces would be invisible to it?

</div>

_Your notes (optional):_

### 1.3 Reconstruction error

A toy 2-D dataset with three outliers appended at the end (rows 100–102).

In [ ]:
np.random.seed(42)
outliers = np.array([[4, -3], [2.8, -3], [-3, 3]])
x1 = np.random.randn(100)
x2 = x1 + np.random.randn(100) / 2
X_toy = np.vstack([np.stack([x1, x2]).T, outliers])
X_toy_scaled = StandardScaler().fit_transform(X_toy)
plt.figure(figsize=(4.5, 4.5))
plt.scatter(X_toy_scaled[:100, 0], X_toy_scaled[:100, 1], c=TEAL, edgecolors="k", lw=0.4)
plt.scatter(X_toy_scaled[100:, 0], X_toy_scaled[100:, 1], c=RED, edgecolors="k", s=60,
            label="outliers")
plt.gca().set_aspect("equal"); plt.legend(); plt.show()

<div class="alert alert-info">

### 1.3.1 &nbsp;·&nbsp; *1 point*

1. Fit scikit-learn's `PCA` with `n_components=1` and `random_state=123` on `X_toy_scaled`.
2. Reconstruct the data (`inverse_transform` of `transform`) to get `X_hat_toy`.
3. Store `get_recon_error_df(X_toy_scaled, X_hat_toy)` in `recon_df`.

**Graded variables** (keep these names): `recon_df`

</div>

In [ ]:
...
recon_df = ...
recon_df.tail()

In [ ]:
check("1.3.1", len(recon_df), recon_df["recon_error"].values[-3:])  # self-check

<div class="alert alert-info">

### 1.3.2 &nbsp;·&nbsp; *2 points*

Compute the average normalised reconstruction error of the three outliers and of the other
100 points (floats).

**Graded variables** (keep these names): `outliers_avg_recon_error`, `non_outliers_avg_recon_error`

</div>

In [ ]:
outliers_avg_recon_error = ...
non_outliers_avg_recon_error = ...
print(f"outliers {outliers_avg_recon_error:.3f}   others {non_outliers_avg_recon_error:.3f}")

In [ ]:
check("1.3.2", outliers_avg_recon_error, non_outliers_avg_recon_error)  # self-check

<div class="alert alert-success">

🧠 **Think about it** — *not graded; questions like this may appear on Tests 2 and 3.*

Why are the outliers reconstructed so badly by one component, and what could you use that for?

</div>

_Your notes (optional):_

<div class="alert alert-warning">⚠️ Good moment to commit.</div>

---
## Exercise 2 — Implementing PCA with the SVD

<div class="alert alert-info">

### 2 `MyPCA` &nbsp;·&nbsp; *8 points*

`MyPCA` below already has `__init__` and `fit`. `fit` centres the data and takes the SVD
$X - \mu = U S V^\top$; the first $k$ rows of $V^\top$ are the components. Complete the four
remaining methods (2 points each):

1. `get_components()` returns $W$ ($k \times d$).
2. `transform(X)` returns $Z = (X-\mu)W^\top$. *Centre with the mean learned in `fit`, not the
   mean of the `X` you are given.*
3. `reconstruct(Z)` returns $\hat X = ZW + \mu$.
4. `explained_variance_ratio()` returns a length-$k$ array: the fraction of the total variance
   captured by each kept component. *Hint: the variance along component $i$ is proportional to
   $s_i^2$, and the total is proportional to $\sum_j s_j^2$ over **all** singular values (L9,
   slide 9).*

The test cell compares your class with scikit-learn for every $k$ from 1 to 20. The marking
program runs similar tests on other data, including more features than examples.

**Graded variables** (keep these names): `MyPCA`

</div>

In [ ]:
class MyPCA:
    """PCA via the SVD of the centred data matrix."""

    def __init__(self, k):
        self.k = k

    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        U, S, Vt = np.linalg.svd(X - self.mean, full_matrices=False)
        self.S = S                 # all singular values, largest first
        self.W = Vt[: self.k, :]   # first k right singular vectors = components (k x d)
        return self

    def get_components(self):
        ...

    def transform(self, X):
        ...

    def reconstruct(self, Z):
        ...

    def explained_variance_ratio(self):
        ...

In [ ]:
from sklearn.datasets import make_blobs

X_blobs, _ = make_blobs(n_samples=100, centers=3, n_features=20, random_state=3950)
X_test, _ = make_blobs(n_samples=30, centers=3, n_features=20, random_state=7)

for k_ in range(1, X_blobs.shape[1] + 1):
    pca_ = PCA(n_components=k_).fit(X_blobs)
    mine = MyPCA(k_).fit(X_blobs)

    # components and scores are defined only up to sign, so compare absolute values
    assert np.allclose(np.abs(pca_.components_), np.abs(mine.get_components())), f"k={k_}: W differs"
    assert np.allclose(np.abs(pca_.transform(X_test)), np.abs(mine.transform(X_test))), \
        f"k={k_}: Z differs (are you centring with the training mean?)"
    # the reconstruction does not depend on the signs, so compare it directly
    assert np.allclose(pca_.inverse_transform(pca_.transform(X_test)),
                       mine.reconstruct(mine.transform(X_test))), f"k={k_}: X_hat differs"
    assert np.allclose(pca_.explained_variance_ratio_, mine.explained_variance_ratio()), \
        f"k={k_}: explained variance ratio differs"
print("All tests passed for k = 1 ... 20 ✓")

<div class="alert alert-warning">⚠️ Good moment to commit.</div>

---
## Exercise 3 — Eigenfaces on animal faces

The data are 1,500 grayscale animal faces, 100×100 pixels, drawn from the **Animal Faces-HQ**
dataset (Choi et al., *StarGAN v2*, CVPR 2020; CC BY-NC 4.0) as curated for the original
version of this assignment. The faces are roughly aligned: eyes near the same height, the face
filling the frame. That alignment is what makes pixel-wise PCA meaningful.

In [ ]:
animals = load_animal_faces()
print(animals.shape, animals.dtype, f"range [{animals.min():.0f}, {animals.max():.0f}]")
show_images(animals[:10], titles=[f"#{i}" for i in range(10)], vmin=0, vmax=1)

Flatten each image into a row of 10,000 pixels. Use `X_anims` for the rest of the assignment.

In [ ]:
X_anims = animals.reshape(len(animals), -1)
image_shape = (100, 100)
X_anims.shape

<div class="alert alert-info">

### 3.1 Choosing the number of components &nbsp;·&nbsp; *9 points*

1. Fit scikit-learn's `PCA` on `X_anims` with `n_components=1000` and `random_state=42`
   (about 10 s). Store the **cumulative** explained variance for $k = 1,\dots,1000$ in
   `cum_var` (length-1000 array). Plot it. *(2)*
2. Store the smallest $k$ reaching at least 90% and 95% of the variance in `k90` and `k95`
   (integers). *(2)*
3. For $k$ = 10, 50, 100, 200, 300 and 500, reconstruct **all 1,500** faces from the first $k$
   components and compute the **mean PSNR** over the faces, where for each face
   $\mathrm{PSNR} = 10\log_{10}(1/\mathrm{MSE})$ (pixels are in $[0, 1]$). Store a dict
   `psnr_by_k` mapping each $k$ (int) to its mean PSNR in dB (float). Also show five faces
   at every $k$ (not graded, but look at them). *(3)*
   *Tip: no need to refit for each $k$: the first $k$ rows of `components_` from the
   1000-component fit are the $k$-component solution.*
4. Storing the faces as PCA codes with $k = 200$ means storing $Z$ ($n \times k$), $W$ and
   $\mu$. Store in `storage_ratio` the total count of numbers stored divided by the count in
   the original $X$ (a float). *(2)*

*Do not standardise the pixels: `PCA` centres the data, and every pixel already has the same
scale.*

**Graded variables** (keep these names): `cum_var`, `k90`, `k95`, `psnr_by_k`, `storage_ratio`

</div>

In [ ]:
# 1-2. cumulative explained variance, k90, k95
...

In [ ]:
# 3. reconstructions and mean PSNR
...

In [ ]:
# 4. storage ratio for k = 200
storage_ratio = ...

<div class="alert alert-success">

🧠 **Think about it** — *not graded; questions like this may appear on Tests 2 and 3.*

Which visual content comes back first as $k$ grows, and which last? How many components would
you keep, and why? In `storage_ratio`, which part dominates — the codes or the basis — and
what does that tell you about when PCA compression pays off?

</div>

_Your notes (optional):_

<div class="alert alert-info">

### 3.2 Inputs that do not belong &nbsp;·&nbsp; *7 points*

A PCA model only knows the subspace its training faces span: every input is reconstructed as
*some* combination of animal-face components. The next cell gives you a train/test split and
a dictionary `probes` of ten inputs: a held-out face, three edits of it, a real cat photo framed
differently, a human face, a cartoon cat, a photograph that is not a face, flat grey, and noise.

1. Fit `PCA(n_components=300, random_state=42)` on the **training** faces only
   (`X_anims[train_idx]`). Show each probe above its reconstruction (not graded).
2. Store the reconstruction RMSE, $\sqrt{\text{mean squared pixel error}}$, of each **test**
   face in `rmse_test` (array of length 150, in `test_idx` order). *(2)*
3. Store the 95th percentile of `rmse_test` in `rmse_thresh` (`np.percentile`). *(1)*
4. Store a dict `rmse_probe` mapping each probe name to its RMSE. *(2)*
5. Store in `flagged_probes` the **list of probe names** whose RMSE is above `rmse_thresh`. *(1)*
6. Based on your results, what does reconstruction error at $k = 300$ mainly measure? Set
   `recon_error_measures` to (A) how much the image looks like an animal face,
   (B) how much fine detail and texture the image has, or (C) how bright the image is. *(1)*

**Graded variables** (keep these names): `rmse_test`, `rmse_thresh`, `rmse_probe`, `flagged_probes`, `recon_error_measures`

</div>

In [ ]:
from skimage import color, data, draw, transform

rng = np.random.default_rng(3950)
perm = rng.permutation(len(X_anims))
train_idx, test_idx = perm[:1350], perm[1350:]

def _to_face(img, box=None):
    img = color.rgb2gray(img) if img.ndim == 3 else img / 255.0
    if box is not None:
        r0, r1, c0, c1 = box; img = img[r0:r1, c0:c1]
    return transform.resize(img, image_shape, anti_aliasing=True).astype(np.float32)

def _cartoon_cat():
    im = np.full(image_shape, 0.95, np.float32)
    for xs in ([18, 22, 42], [82, 78, 58]):                       # ears
        rr, cc = draw.polygon([45, 10, 30], xs); im[rr, cc] = 0.55
    rr, cc = draw.disk((58, 50), 36, shape=image_shape); im[rr, cc] = 0.55   # head
    for cx in (36, 64):                                              # eyes
        rr, cc = draw.disk((52, cx), 7, shape=image_shape); im[rr, cc] = 1.0
        rr, cc = draw.disk((53, cx), 3.5, shape=image_shape); im[rr, cc] = 0.05
    rr, cc = draw.polygon([66, 66, 72], [45, 55, 50]); im[rr, cc] = 0.2     # nose
    for s in (-1, 1):                                                # whiskers
        for dy in (-3, 2, 7):
            rr, cc = draw.line(70, 50 + s * 8, 70 + dy, 50 + s * 30); im[rr, cc] = 0.1
    return im

face = animals[test_idx[43]]   # a typical, well-aligned held-out face
probes = {
    "held-out face":        face,
    "shifted 10 px":        np.roll(face, 10, axis=1),
    "upside down":          face[::-1],
    "darkened (γ = 3)":     face ** 3,
    "real cat, new framing": _to_face(data.chelsea(), (0, 300, 75, 375)),
    "human face":           _to_face(data.astronaut(), (10, 210, 130, 330)),
    "cartoon cat":          _cartoon_cat(),
    "not a face":           _to_face(data.camera()),
    "flat grey":            np.full(image_shape, 0.5, np.float32),
    "noise":                rng.random(image_shape).astype(np.float32),
}
print(f"{len(train_idx)} training faces, {len(test_idx)} test faces, {len(probes)} probes")
show_images(list(probes.values()), titles=list(probes), ncol=5, vmin=0, vmax=1)

In [ ]:
...

<div class="alert alert-success">

🧠 **Think about it** — *not graded; questions like this may appear on Tests 2 and 3.*

What artefacts appear in the reconstructions of the probes that are not animal faces? Why does
flat grey get the lowest error of all? If this PCA model were a preprocessing step in a larger
system, what would you check before trusting an input's projection?

</div>

_Your notes (optional):_

<div class="alert alert-warning">⚠️ Good moment to commit.</div>

---
## Exercise 4 — Interpreting the components

The cell below fits PCA with 300 components on all 1,500 faces and keeps $Z$, $W$ and $\hat X$
for the questions that follow.

In [ ]:
pca = PCA(n_components=300, random_state=42).fit(X_anims)
Z = pca.transform(X_anims)
W = pca.components_
X_hat = pca.inverse_transform(Z)
print("Z", Z.shape, "  W", W.shape, "  X_hat", X_hat.shape)

<div class="alert alert-info">

### 4.1 What the components look like &nbsp;·&nbsp; *6 points*

1. Show the first 15 components as 100×100 images in a 3×5 grid with a diverging colormap
   (`cmap="RdBu_r"`) and limits symmetric about zero (not graded, but you need it for part 3).
2. Write `strongest(Z, c, n=6, positive=True)`: return the indices (NumPy int array, length
   `n`) of the `n` rows of `Z` with the **largest** values in column `c`, largest first; with
   `positive=False`, the `n` **most negative**, most negative first. The helper
   `plot_strong_comp_images` uses your function. *(3)*
3. Use `plot_strong_comp_images` on components 0 to 7, both directions. Then identify which
   component (index 0–14) best matches each description. *(3)*
   * `brightness_component`: overall brightness (pale animals on light backgrounds at one end,
 dark animals on dark backgrounds at the other)
   * `figure_ground_component`: the face against its background (dark face on light background
 at one end, light face on dark background at the other)
   * `left_right_component`: the left half against the right half (side lighting)

Signs are arbitrary: the two ends of a component matter, not which is called positive.

**Graded variables** (keep these names): `strongest`, `brightness_component`, `figure_ground_component`, `left_right_component`

</div>

In [ ]:
# 1. first 15 components
...

In [ ]:
# 2. strongest
def strongest(Z, c, n=6, positive=True):
    ...

In [ ]:
# 3. look, then identify
for c in range(8):
    plot_strong_comp_images(X_anims, Z, W, compn=c)
    plot_strong_comp_images(X_anims, Z, W, compn=c, positive_direction=False)

brightness_component = ...
figure_ground_component = ...
left_right_component = ...

<div class="alert alert-success">

🧠 **Think about it** — *not graded; questions like this may appear on Tests 2 and 3.*

What do the leading components describe: the animal, or how the photo was taken? What does
that predict about using PCA codes to recognise species, or to match photos taken under
different lighting?

</div>

_Your notes (optional):_

<div class="alert alert-info">

### 4.2 Checking the interpretation with numbers &nbsp;·&nbsp; *3 points*

Interpretations by eye can fool you. Test them: for each face compute

* `brightness` — the mean of all its pixels;
* `figure_ground` — the mean of its **border** pixels (the outer 15 pixels on every side)
  minus the mean of the remaining centre pixels;
* `left_right` — the mean of its left half (columns 0–49) minus the mean of its right half;

and store the **Pearson correlation** (`np.corrcoef`) between each measure and the score on the
component you named in 4.1: `r_brightness` (with `Z[:, brightness_component]`),
`r_figure_ground` and `r_left_right` (floats). One point each. Signs depend on the arbitrary
component signs, so only the magnitude is marked. A scatter plot of the first two columns of
`Z` is a good way to see the face space (not graded).

**Graded variables** (keep these names): `r_brightness`, `r_figure_ground`, `r_left_right`

</div>

In [ ]:
...

<div class="alert alert-info">

### 4.3 Walking through face space &nbsp;·&nbsp; *7 points*

1. Make two grids of 5 values with `np.linspace` from the **5th to the 95th percentile** of
   `Z[:, 0]` and of `Z[:, 1]`. For each grid point $(a, b)$ find the face whose
   $(z_1, z_2)$ is **closest** (Euclidean) and store its index in `tile_idx[i, j]`, where `i`
   indexes the first-dimension grid and `j` the second (`tile_idx`: 5×5 integer array). Show it with
   `img_tiling(X_anims, tile_idx)` and label the axes with your interpretation. *(5: one fifth
   of a point per correct cell)*
2. Across your 5×5 grid, which attribute changes **least systematically**? Set
   `least_systematic` to (A) brightness, (B) figure/ground contrast, or (C) species. *(2)*

**Graded variables** (keep these names): `tile_idx`, `least_systematic`

</div>

In [ ]:
...

<div class="alert alert-success">

🧠 **Think about it** — *not graded; questions like this may appear on Tests 2 and 3.*

Describe what changes left to right and bottom to top in your tiling. Does it agree with 4.1
and 4.2? Why is species scattered across the grid?

</div>

_Your notes (optional):_

<div class="alert alert-warning">⚠️ Good moment to commit.</div>

---
## Exercise 5 — Challenge: searching in face space

*Attempt this after everything else. It is worth 4 of the 60 points.*

<div class="alert alert-info">

### 5 Is face space a better place to search than pixel space? &nbsp;·&nbsp; *4 points*

A common use of PCA is **retrieval**: store every image as its code $z$, and answer a query by
returning the stored image whose code is nearest. The cell below makes a gallery (all 1,500
faces) and, for 300 of them, perturbed copies to use as queries. A query is answered correctly
if its nearest gallery image is the face it came from.

1. Write `top1_accuracy(gallery_codes, query_codes, true_idx)` returning the **fraction**
   (0–1) of queries whose nearest gallery row (Euclidean distance) is `true_idx[q]`. Avoid a
   Python loop over pairs: $\|a-b\|^2 = \|a\|^2 - 2a\cdot b + \|b\|^2$ gives every distance
   with one matrix product. *(2)*
2. Build a pandas DataFrame `retrieval_table` of top-1 accuracy **in percent**, one row per
   perturbation (use the keys of `queries` as the index) and at least these two columns:
   `"pixels"` (raw pixels) and `"PCA k=50"` (the first 50 scores of
   `PCA(n_components=100, random_state=42)` fitted on all of `X_anims`). Add at least one
   column of your own design (ideas: drop the first few components; divide each score by
   $\sqrt{\lambda_i}$; a different $k$). *(2: the two required columns)*

**Graded variables** (keep these names): `top1_accuracy`, `retrieval_table`

</div>

In [ ]:
from scipy.ndimage import gaussian_filter

rng5 = np.random.default_rng(5)
query_idx = rng5.choice(len(animals), 300, replace=False)
perturbations = {
    "none":            lambda im: im,
    "noise σ=0.15":    lambda im: np.clip(im + rng5.normal(0, 0.15, im.shape), 0, 1),
    "darker (γ=2)":    lambda im: im ** 2,
    "shift 4 px":      lambda im: np.roll(im, 4, axis=1),
    "shift 8 px":      lambda im: np.roll(im, 8, axis=1),
    "blur σ=2":        lambda im: gaussian_filter(im, 2),
    "mirror":          lambda im: im[:, ::-1],
    "top 35% hidden":  lambda im: np.where(np.arange(100)[:, None] < 35, 0.0, im),
}
queries = {name: np.stack([f(animals[i]) for i in query_idx]).reshape(len(query_idx), -1)
           .astype(np.float32) for name, f in perturbations.items()}
show_images([queries[n][0] for n in perturbations], titles=list(perturbations), ncol=8, size=1.6,
            vmin=0, vmax=1)

In [ ]:
def top1_accuracy(gallery_codes, query_codes, true_idx):
    ...

...

<div class="alert alert-success">

🧠 **Think about it** — *not graded; questions like this may appear on Tests 2 and 3.*

Which perturbations does PCA handle better than raw pixels, and which worse? Explain the
brightness result using what you found in Exercise 4. What would you do about the
perturbations that defeat every representation?

</div>

_Your notes (optional):_

---
## AI disclosure *(Exercise 0, 1 point)*

In the cell below, replace `...` with a string listing every generative-AI tool you used and
what for, or `"None"`. Example: `"Claude: explained why my transform failed the Z test
(I was centring with the test mean). ChatGPT: matplotlib syntax for 4.1."`

In [ ]:
ai_disclosure = ...

## Submission checklist

- [ ] Restarted and ran all cells; nothing crashes
- [ ] Every graded variable is set (no `...` left)
- [ ] `ai_disclosure` filled in
- [ ] At least three commits, the last one pushed before the deadline
- [ ] No data files committed

---
<sub>Adapted for MCS 3950 from *Dimensionality Reduction Adventures with Animal Faces* by
Varada Kolhatkar (UBC Master of Data Science; EAAI-2026 model assignment), used under
CC BY 4.0. Changes: course-specific setup and automatic marking; reworked off-distribution
exercise (3.2); quantitative checks of component interpretations (4.2); the CNN autoencoder
extension replaced by a retrieval challenge (5). Images: Animal Faces-HQ (Choi et al., 2020),
CC BY-NC 4.0; probe images from scikit-image.</sub>